# 🎓 Tahap 1 — Manual Fuzzy Inference System (Mamdani)
## Prediksi Risiko Dropout Mahasiswa

| | |
|---|---|
| **Mata Kuliah** | Soft Computing |
| **Dosen** | Dr. Ir. Intan Nurma Yulita, M.T |
| **FIS Type** | Mamdani |
| **Operator AND** | Minimum |
| **Operator OR** | Maximum |
| **Defuzzifikasi** | Centroid (Center of Gravity) |
| **Dataset** | UCI — Students Dropout and Academic Success |

---
### Alur Notebook
1. Install & Import Library
2. Definisi Membership Functions (MF)
3. Visualisasi MF
4. Rule Base (30 Rules)
5. Demo Prediksi Single Instance
6. Evaluasi Batch (n=120)
7. Ringkasan Performa (Baseline)

---
## 1. Install & Import Library

In [ ]:
!pip install scikit-fuzzy matplotlib numpy scipy -q

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Library berhasil diimport')
print(f'   numpy   : {np.__version__}')
print(f'   skfuzzy : {fuzz.__version__}')

---
## 2. Definisi Membership Functions

### Variabel Input
| # | Variabel | Range | Label Linguistik | Tipe MF |
|---|---|---|---|---|
| x1 | IPK Semester | 0.0 - 4.0 | Rendah / Sedang / Tinggi | Trapesium / Segitiga |
| x2 | Tingkat Kehadiran | 0 - 100% | Jarang / Cukup / Rajin | Trapesium / Segitiga |
| x3 | Jumlah MK Gagal | 0 - 10 | Sedikit / Sedang / Banyak | Trapesium / Segitiga |
| x4 | Status Ekonomi | 0.0 - 1.0 | Rentan / Stabil | Gaussian |

### Variabel Output
| # | Variabel | Range | Label Linguistik | Defuzzifikasi |
|---|---|---|---|---|
| y | Risiko Dropout | 0 - 100 | Rendah / Sedang / Tinggi | Centroid |

In [ ]:
# Antecedent (Input Variables)
ipk         = ctrl.Antecedent(np.arange(0, 4.01, 0.01),  'ipk')
kehadiran   = ctrl.Antecedent(np.arange(0, 101, 1),       'kehadiran')
mk_gagal    = ctrl.Antecedent(np.arange(0, 10.1, 0.1),    'mk_gagal')
status_ekon = ctrl.Antecedent(np.arange(0, 1.01, 0.01),   'status_ekon')

# Consequent (Output Variable)
risiko = ctrl.Consequent(np.arange(0, 101, 1), 'risiko', defuzzify_method='centroid')

# MF: IPK
ipk['Rendah'] = fuzz.trapmf(ipk.universe, [0,   0,   1.5, 2.2])
ipk['Sedang'] = fuzz.trimf (ipk.universe, [1.8, 2.5, 3.2])
ipk['Tinggi'] = fuzz.trapmf(ipk.universe, [2.8, 3.3, 4.0, 4.0])

# MF: Kehadiran
kehadiran['Jarang'] = fuzz.trapmf(kehadiran.universe, [0,  0,  40, 60])
kehadiran['Cukup']  = fuzz.trimf (kehadiran.universe, [50, 70, 85])
kehadiran['Rajin']  = fuzz.trapmf(kehadiran.universe, [75, 85, 100, 100])

# MF: MK Gagal
mk_gagal['Sedikit'] = fuzz.trapmf(mk_gagal.universe, [0, 0, 1,  3])
mk_gagal['Sedang']  = fuzz.trimf (mk_gagal.universe, [2, 4, 6])
mk_gagal['Banyak']  = fuzz.trapmf(mk_gagal.universe, [5, 7, 10, 10])

# MF: Status Ekonomi
status_ekon['Rentan'] = fuzz.gaussmf(status_ekon.universe, 0,   0.15)
status_ekon['Stabil'] = fuzz.gaussmf(status_ekon.universe, 1.0, 0.15)

# MF: Output Risiko Dropout
risiko['Rendah'] = fuzz.trapmf(risiko.universe, [0,  0,  25, 45])
risiko['Sedang'] = fuzz.trimf (risiko.universe, [35, 50, 65])
risiko['Tinggi'] = fuzz.trapmf(risiko.universe, [55, 75, 100, 100])

print('Membership Functions berhasil didefinisikan')
print('  Input variables  : 4')
print('  Output variables : 1')
print('  Total MF         : 14')

---
## 3. Visualisasi Membership Functions

In [ ]:
COLORS = {
    'Rendah' : '#1D9E75', 'Sedang' : '#EF9F27', 'Tinggi' : '#E24B4A',
    'Jarang' : '#E24B4A', 'Cukup'  : '#EF9F27', 'Rajin'  : '#1D9E75',
    'Sedikit': '#1D9E75', 'Banyak' : '#E24B4A',
    'Rentan' : '#E24B4A', 'Stabil' : '#1D9E75',
}

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Membership Functions - Tahap 1: Manual FIS (Mamdani)\n'
             'Prediksi Risiko Dropout Mahasiswa',
             fontsize=14, fontweight='bold', y=0.98)

gs   = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.35)
axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(3)]

vars_info = [
    (ipk,         'x1 - IPK Semester',           'Nilai IPK'),
    (kehadiran,   'x2 - Tingkat Kehadiran',       'Kehadiran (%)'),
    (mk_gagal,    'x3 - Jumlah MK Gagal',         'Jumlah MK'),
    (status_ekon, 'x4 - Status Ekonomi',          'Encoded (0=Rentan, 1=Stabil)'),
    (risiko,      'y  - Risiko Dropout (Output)', 'Skor Risiko (0-100)'),
]

for ax, (var, title, xlabel) in zip(axes, vars_info):
    for label in var.terms:
        color = COLORS.get(label, '#888888')
        ax.plot(var.universe, var[label].mf, label=label, color=color, linewidth=2.2)
        ax.fill_between(var.universe, var[label].mf, alpha=0.10, color=color)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel('mu (derajat keanggotaan)', fontsize=8)
    ax.set_ylim(-0.05, 1.18)
    ax.legend(fontsize=8, loc='upper right', framealpha=0.7)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.spines[['top', 'right']].set_visible(False)

axes[5].axis('off')
info_text = (
    'Konfigurasi FIS\n'
    '-----------------\n'
    'Type   : Mamdani\n'
    'AND op : Minimum\n'
    'OR op  : Maximum\n'
    'Defuzz : Centroid\n'
    '-----------------\n'
    'Input  : 4 var\n'
    'Output : 1 var\n'
    'Total MF : 14\n'
    'Rules  : 30'
)
axes[5].text(0.5, 0.52, info_text, ha='center', va='center', fontsize=10.5,
             fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.7', facecolor='#EEF2FF',
                       edgecolor='#AABBDD', linewidth=1.5),
             transform=axes[5].transAxes)

plt.savefig('mf_tahap1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan: mf_tahap1.png')

---
## 4. Rule Base - 30 Rules (Human Expert Intuition)

> **Format:** `IF (x1 is A) AND (x2 is B) AND (x3 is C) AND (x4 is D) THEN y is Z`
>
> **AND:** Minimum &nbsp;|&nbsp; **Agregasi OR:** Maximum &nbsp;|&nbsp; **Defuzz:** Centroid

| Kelompok | Jumlah | Deskripsi |
|---|---|---|
| Risiko Tinggi | 10 | Kombinasi indikator akademik & ekonomi buruk |
| Risiko Sedang | 12 | Indikator campuran, perlu monitoring |
| Risiko Rendah | 8  | Indikator akademik baik |
| **Total** | **30** | |

In [ ]:
# RISIKO TINGGI (10 rules)
r01 = ctrl.Rule(ipk['Rendah'] & kehadiran['Jarang'] & mk_gagal['Banyak']  & status_ekon['Rentan'], risiko['Tinggi'])
r02 = ctrl.Rule(ipk['Rendah'] & kehadiran['Jarang'] & mk_gagal['Banyak']  & status_ekon['Stabil'], risiko['Tinggi'])
r03 = ctrl.Rule(ipk['Rendah'] & kehadiran['Jarang'] & mk_gagal['Sedang']  & status_ekon['Rentan'], risiko['Tinggi'])
r04 = ctrl.Rule(ipk['Rendah'] & kehadiran['Cukup']  & mk_gagal['Banyak']  & status_ekon['Rentan'], risiko['Tinggi'])
r05 = ctrl.Rule(ipk['Rendah'] & kehadiran['Jarang'] & mk_gagal['Sedang']  & status_ekon['Stabil'], risiko['Tinggi'])
r06 = ctrl.Rule(ipk['Rendah'] & kehadiran['Cukup']  & mk_gagal['Banyak']  & status_ekon['Stabil'], risiko['Tinggi'])
r07 = ctrl.Rule(ipk['Sedang'] & kehadiran['Jarang'] & mk_gagal['Banyak']  & status_ekon['Rentan'], risiko['Tinggi'])
r08 = ctrl.Rule(ipk['Rendah'] & kehadiran['Rajin']  & mk_gagal['Banyak']  & status_ekon['Rentan'], risiko['Tinggi'])
r09 = ctrl.Rule(ipk['Sedang'] & kehadiran['Jarang'] & mk_gagal['Banyak']  & status_ekon['Stabil'], risiko['Tinggi'])
r10 = ctrl.Rule(ipk['Rendah'] & kehadiran['Jarang'] & mk_gagal['Sedikit'] & status_ekon['Rentan'], risiko['Tinggi'])

# RISIKO SEDANG (12 rules)
r11 = ctrl.Rule(ipk['Rendah'] & kehadiran['Cukup']  & mk_gagal['Sedang']  & status_ekon['Stabil'], risiko['Sedang'])
r12 = ctrl.Rule(ipk['Sedang'] & kehadiran['Cukup']  & mk_gagal['Sedang']  & status_ekon['Rentan'], risiko['Sedang'])
r13 = ctrl.Rule(ipk['Sedang'] & kehadiran['Jarang'] & mk_gagal['Sedang']  & status_ekon['Stabil'], risiko['Sedang'])
r14 = ctrl.Rule(ipk['Sedang'] & kehadiran['Jarang'] & mk_gagal['Sedikit'] & status_ekon['Rentan'], risiko['Sedang'])
r15 = ctrl.Rule(ipk['Rendah'] & kehadiran['Rajin']  & mk_gagal['Sedang']  & status_ekon['Stabil'], risiko['Sedang'])
r16 = ctrl.Rule(ipk['Rendah'] & kehadiran['Cukup']  & mk_gagal['Sedikit'] & status_ekon['Rentan'], risiko['Sedang'])
r17 = ctrl.Rule(ipk['Sedang'] & kehadiran['Cukup']  & mk_gagal['Banyak']  & status_ekon['Stabil'], risiko['Sedang'])
r18 = ctrl.Rule(ipk['Sedang'] & kehadiran['Jarang'] & mk_gagal['Sedang']  & status_ekon['Rentan'], risiko['Sedang'])
r19 = ctrl.Rule(ipk['Tinggi'] & kehadiran['Jarang'] & mk_gagal['Sedang']  & status_ekon['Rentan'], risiko['Sedang'])
r20 = ctrl.Rule(ipk['Rendah'] & kehadiran['Rajin']  & mk_gagal['Sedikit'] & status_ekon['Rentan'], risiko['Sedang'])
r21 = ctrl.Rule(ipk['Sedang'] & kehadiran['Rajin']  & mk_gagal['Sedang']  & status_ekon['Rentan'], risiko['Sedang'])
r22 = ctrl.Rule(ipk['Tinggi'] & kehadiran['Cukup']  & mk_gagal['Banyak']  & status_ekon['Rentan'], risiko['Sedang'])

# RISIKO RENDAH (8 rules)
r23 = ctrl.Rule(ipk['Tinggi'] & kehadiran['Rajin']  & mk_gagal['Sedikit'] & status_ekon['Stabil'], risiko['Rendah'])
r24 = ctrl.Rule(ipk['Tinggi'] & kehadiran['Rajin']  & mk_gagal['Sedikit'] & status_ekon['Rentan'], risiko['Rendah'])
r25 = ctrl.Rule(ipk['Tinggi'] & kehadiran['Cukup']  & mk_gagal['Sedikit'] & status_ekon['Stabil'], risiko['Rendah'])
r26 = ctrl.Rule(ipk['Sedang'] & kehadiran['Rajin']  & mk_gagal['Sedikit'] & status_ekon['Stabil'], risiko['Rendah'])
r27 = ctrl.Rule(ipk['Tinggi'] & kehadiran['Rajin']  & mk_gagal['Sedang']  & status_ekon['Stabil'], risiko['Rendah'])
r28 = ctrl.Rule(ipk['Sedang'] & kehadiran['Rajin']  & mk_gagal['Sedikit'] & status_ekon['Rentan'], risiko['Rendah'])
r29 = ctrl.Rule(ipk['Tinggi'] & kehadiran['Cukup']  & mk_gagal['Sedikit'] & status_ekon['Rentan'], risiko['Rendah'])
r30 = ctrl.Rule(ipk['Sedang'] & kehadiran['Cukup']  & mk_gagal['Sedikit'] & status_ekon['Stabil'], risiko['Rendah'])

# Build FIS
all_rules = [
    r01, r02, r03, r04, r05, r06, r07, r08, r09, r10,
    r11, r12, r13, r14, r15, r16, r17, r18, r19, r20, r21, r22,
    r23, r24, r25, r26, r27, r28, r29, r30
]
fis_ctrl = ctrl.ControlSystem(all_rules)
sim      = ctrl.ControlSystemSimulation(fis_ctrl)

print('Rule Base berhasil dimuat')
print('  Risiko Tinggi : 10 rules')
print('  Risiko Sedang : 12 rules')
print('  Risiko Rendah :  8 rules')
print('  Total         : 30 rules')

---
## 5. Demo Prediksi - Single Instance

Uji 5 skenario mahasiswa dengan karakteristik berbeda.

In [ ]:
def predict(sim, ipk_val, kehadiran_val, mk_gagal_val, status_ekon_val):
    sim.input['ipk']         = float(ipk_val)
    sim.input['kehadiran']   = float(kehadiran_val)
    sim.input['mk_gagal']    = float(mk_gagal_val)
    sim.input['status_ekon'] = float(status_ekon_val)
    sim.compute()
    score = sim.output['risiko']
    label = 'Rendah' if score < 40 else ('Sedang' if score < 65 else 'Tinggi')
    return round(score, 2), label

scenarios = [
    ('Mahasiswa Ideal',        3.8, 95, 0,  0.95),
    ('Risiko Sedang',          2.4, 68, 3,  0.40),
    ('Dropout Kritis',         1.1, 22, 8,  0.05),
    ('IPK Baik tapi Absen',    3.2, 38, 2,  0.80),
    ('Ekonomi Lemah + IPK OK', 2.9, 80, 1,  0.10),
]

print(f'{"Skenario":<28} {"IPK":>5} {"Had%":>6} {"MKF":>5} {"Eko":>5}  {"Skor":>7}  Label')
print('-' * 72)
for name, i, h, m, e in scenarios:
    score, label = predict(sim, i, h, m, e)
    print(f'{name:<28} {i:>5} {h:>6} {m:>5} {e:>5}  {score:>6.1f}  {label}')
print('-' * 72)

In [ ]:
scores = [predict(sim, *s[1:])[0] for s in scenarios]
labels = [predict(sim, *s[1:])[1] for s in scenarios]
names  = [s[0] for s in scenarios]
colors = ['#E24B4A' if l=='Tinggi' else '#EF9F27' if l=='Sedang' else '#1D9E75' for l in labels]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(names, scores, color=colors, height=0.55, edgecolor='white', linewidth=1.2)
ax.axvline(40, color='#EF9F27', linestyle='--', linewidth=1, alpha=0.7, label='Batas Sedang (40)')
ax.axvline(65, color='#E24B4A', linestyle='--', linewidth=1, alpha=0.7, label='Batas Tinggi (65)')
for bar, score, label in zip(bars, scores, labels):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}  [{label}]', va='center', fontsize=9)
ax.set_xlim(0, 115)
ax.set_xlabel('Skor Risiko Dropout', fontsize=10)
ax.set_title('Demo Prediksi - 5 Skenario Mahasiswa', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

---
## 6. Evaluasi Batch - Dataset Simulasi UCI (n=120)

In [ ]:
def load_uci_sample(seed=42):
    np.random.seed(seed)
    data = []
    for _ in range(40):
        data.append({'ipk': np.random.uniform(0.5, 2.0),
                     'kehadiran': np.random.uniform(10, 55),
                     'mk_gagal': np.random.uniform(4, 10),
                     'status_ekon': np.random.uniform(0.0, 0.35),
                     'label_true': 'Tinggi'})
    for _ in range(40):
        data.append({'ipk': np.random.uniform(1.8, 3.0),
                     'kehadiran': np.random.uniform(45, 78),
                     'mk_gagal': np.random.uniform(1, 5),
                     'status_ekon': np.random.uniform(0.2, 0.75),
                     'label_true': 'Sedang'})
    for _ in range(40):
        data.append({'ipk': np.random.uniform(2.8, 4.0),
                     'kehadiran': np.random.uniform(72, 100),
                     'mk_gagal': np.random.uniform(0, 2),
                     'status_ekon': np.random.uniform(0.5, 1.0),
                     'label_true': 'Rendah'})
    return data

def evaluate(sim, dataset):
    label_order = ['Rendah', 'Sedang', 'Tinggi']
    confusion = {t: {p: 0 for p in label_order} for t in label_order}
    correct, results = 0, []
    for row in dataset:
        try:
            score, pred = predict(sim, row['ipk'], row['kehadiran'],
                                  row['mk_gagal'], row['status_ekon'])
            true_l = row['label_true']
            confusion[true_l][pred] += 1
            if pred == true_l:
                correct += 1
            results.append({'true': true_l, 'pred': pred, 'score': score})
        except Exception:
            pass
    n = len(results)
    accuracy  = correct / n if n > 0 else 0
    per_class = {l: confusion[l][l] / max(sum(confusion[l].values()), 1)
                 for l in label_order}
    return {'accuracy': round(accuracy*100, 2), 'n': n,
            'correct': correct, 'confusion': confusion,
            'per_class': per_class}

print('Menjalankan evaluasi...')
dataset = load_uci_sample()
result  = evaluate(sim, dataset)

print(f'Akurasi keseluruhan : {result["accuracy"]}%')
print(f'Benar / Total       : {result["correct"]} / {result["n"]}')
label_order = ['Rendah', 'Sedang', 'Tinggi']
for lbl in label_order:
    acc = result['per_class'][lbl] * 100
    bar = chr(9608) * int(acc / 5)
    print(f'  {lbl:<8}: {acc:>5.1f}%  {bar}')

In [ ]:
label_order = ['Rendah', 'Sedang', 'Tinggi']
cm_array = np.array([[result['confusion'][t][p] for p in label_order]
                      for t in label_order])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Evaluasi Tahap 1 - Manual FIS (Baseline)\n'
             f'Akurasi: {result["accuracy"]}%   |   n = {result["n"]}',
             fontsize=12, fontweight='bold')

im = axes[0].imshow(cm_array, cmap='Blues', vmin=0)
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(label_order)
axes[0].set_yticks(range(3)); axes[0].set_yticklabels(label_order)
axes[0].set_xlabel('Prediksi', fontweight='bold')
axes[0].set_ylabel('Aktual',   fontweight='bold')
axes[0].set_title('Confusion Matrix')
for i in range(3):
    for j in range(3):
        v = cm_array[i, j]
        c = 'white' if v > cm_array.max() * 0.6 else 'black'
        axes[0].text(j, i, str(v), ha='center', va='center',
                     fontsize=14, fontweight='bold', color=c)
plt.colorbar(im, ax=axes[0])

bar_colors = ['#1D9E75', '#EF9F27', '#E24B4A']
vals = [result['per_class'][l]*100 for l in label_order]
bars = axes[1].bar(label_order, vals, color=bar_colors, width=0.5,
                   edgecolor='white', linewidth=1.5)
axes[1].set_ylim(0, 118)
axes[1].set_xlabel('Kelas Risiko', fontweight='bold')
axes[1].set_ylabel('Akurasi (%)',  fontweight='bold')
axes[1].set_title('Akurasi Per Kelas')
axes[1].axhline(result['accuracy'], color='gray', linestyle='--',
                linewidth=1.2, label=f'Overall: {result["accuracy"]}%')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.25)
axes[1].spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('eval_tahap1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan: eval_tahap1.png')

---
## 7. Ringkasan Performa Baseline

Hasil ini menjadi **acuan perbandingan** untuk Tahap 2 (GA) dan Tahap 3 (ANN).

In [ ]:
label_order = ['Rendah', 'Sedang', 'Tinggi']

print('=' * 52)
print('  RINGKASAN PERFORMA - TAHAP 1 (BASELINE)')
print('=' * 52)
print(f'  FIS Type       : Mamdani')
print(f'  Defuzzifikasi  : Centroid')
print(f'  Total Rules    : 30')
print(f'  Dataset        : Simulasi UCI (n=120)')
print()
print(f'  Akurasi Overall: {result["accuracy"]}%')
print(f'  Benar / Total  : {result["correct"]} / {result["n"]}')
print()
print('  Akurasi Per Kelas:')
for lbl in label_order:
    acc = result['per_class'][lbl] * 100
    print(f'    {lbl:<8} : {acc:.1f}%')
print()
print('  Confusion Matrix (baris=Aktual, kolom=Prediksi):')
header = '           ' + '  '.join(f'{l:>8}' for l in label_order)
print(header)
for t in label_order:
    row = f'  {t:<10}' + '  '.join(f'{result["confusion"][t][p]:>8}' for p in label_order)
    print(row)
print('=' * 52)

In [ ]:
import pandas as pd

label_order = ['Rendah', 'Sedang', 'Tinggi']
comparison = pd.DataFrame({
    'Metode'        : ['Tahap 1 - Manual FIS', 'Tahap 2 - GA Tuning', 'Tahap 3 - ANN Tuning'],
    'Akurasi (%)'   : [result['accuracy'], '-', '-'],
    'Acc. Rendah (%)': [f"{result['per_class']['Rendah']*100:.1f}", '-', '-'],
    'Acc. Sedang (%)': [f"{result['per_class']['Sedang']*100:.1f}", '-', '-'],
    'Acc. Tinggi (%)': [f"{result['per_class']['Tinggi']*100:.1f}", '-', '-'],
    'Keterangan'    : ['BASELINE (human expert)', 'Diisi setelah Tahap 2', 'Diisi setelah Tahap 3'],
})
comparison.set_index('Metode', inplace=True)

display(comparison.style
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{'selector': 'th',
                        'props': [('background-color', '#2E75B6'),
                                  ('color', 'white'),
                                  ('text-align', 'center')]}]))